# 08. Un corpus propio: los ojos en la novela en español

En el cuaderno 07 leímos concordancias de *eyes* en Dickens. Todo venía listo: CLiC nos dio el corpus ya dividido en oraciones, con metadatos y con las suspensiones marcadas.

Casi nunca tenemos esa suerte. Si queremos estudiar el español, tenemos que **armar el corpus nosotros**. Eso implica:

1. **Conseguir** los textos.
2. **Limpiarlos**: quitar todo lo que no es parte de la obra.
3. **Anotarlos**: agregar lemas y categorías gramaticales con un **modelo preentrenado**.
4. **Revisar** qué tan bien funcionó ese modelo con nuestros textos.

Después usaremos FlexiConc como en el cuaderno 07, para responder una pregunta que dejamos abierta: en inglés, Dickens escribe *his eyes*. ¿Qué pasa en español, donde decimos *cerró los ojos*?

## 0. Preparación

Instalamos FlexiConc igual que en el cuaderno 07. Además descargamos **`es_core_news_sm`**, un modelo de spaCy para el español. Pesa unos 13 MB.

Si Colab te pide reiniciar la sesión, hazlo y continúa con la celda siguiente.

In [ ]:
!apt-get -qq install -y libicu-dev pkg-config > /dev/null
!pip install -q 'flexiconc[notebooks,web,ICU,partitioning,annotation]'
!python -m spacy download es_core_news_sm

In [ ]:
import re
from pathlib import Path

import pandas as pd
import requests
import spacy

from flexiconc.text_import import TextImport
from flexiconc.utils.notebook_utils import show_kwic, show_analysis_tree

DATOS = Path("datos/novelas_es")
CRUDOS = DATOS / "gutenberg"
CRUDOS.mkdir(parents=True, exist_ok=True)

---
## 1. El corpus

Usaremos cinco novelas del siglo XIX, más o menos contemporáneas de Dickens. Todas están en [Project Gutenberg](https://www.gutenberg.org):

| Archivo | Autor | Novela | Año | País |
|---|---|---|---|---|
| `rivapalacio_monja_1868` | Vicente Riva Palacio | *Monja y casada, virgen y mártir* | 1868 | México |
| `altamirano_navidad_1871` | Ignacio Manuel Altamirano | *La Navidad en las montañas* | 1871 | México |
| `galdos_perfecta_1876` | Benito Pérez Galdós | *Doña Perfecta* | 1876 | España |
| `galdos_marianela_1878` | Benito Pérez Galdós | *Marianela* | 1878 | España |
| `pardobazan_pazos_1886` | Emilia Pardo Bazán | *Los pazos de Ulloa* | 1886 | España |

Galdós es un buen puente con el cuaderno anterior: admiraba a Dickens, y una de sus primeras publicaciones fue una traducción de *The Pickwick Papers* (1868).

**¿Por qué sólo textos tan viejos?** Por los derechos de autor. En México una obra entra al dominio público **100 años después de la muerte** de su autor. Altamirano murió en 1893 y Riva Palacio en 1896, así que sus obras son libres. Para un autor que murió en 1940, habría que esperar hasta 2040. Ésta es una restricción real al armar corpus literarios: lo que está disponible no siempre es lo que más nos interesa estudiar.

En el diccionario `LIBROS` guardamos, para cada novela, su número en Gutenberg y dos patrones que marcan dónde **empieza** y dónde **termina** el texto de la novela. Enseguida veremos por qué hacen falta.

In [ ]:
LIBROS = {
    # nombre: (número en Gutenberg, línea de inicio, línea de fin)
    "rivapalacio_monja_1868":  (49560, r"LIBRO PRIMERO\.", r"ÍNDICE\."),
    "altamirano_navidad_1871": (10825, r"DEDICATORIA A FRANCISCO SOSA", r"VOCABULARY"),
    "galdos_perfecta_1876":    (15725, r"I", r"NOTES"),
    "galdos_marianela_1878":   (17340, r"-I-", None),
    "pardobazan_pazos_1886":   (18005, r"Tomo I", None),
}

---
## 2. Conseguir y limpiar los textos

### Descargar

Gutenberg ofrece cada libro como texto plano en una dirección fija. Descargamos cada uno una sola vez y lo guardamos en `datos/novelas_es/gutenberg/`.

A veces Gutenberg tarda o bloquea descargas automáticas. Por eso, si algo falla, la función toma una copia ya limpia que está en el repositorio del curso.

In [ ]:
RESPALDO = "https://raw.githubusercontent.com/Maclenn77/programacion-para-linguistas/main/datos/novelas_es/"

def descargar(numero):
    destino = CRUDOS / f"pg{numero}.txt"
    if not destino.exists():
        url = f"https://www.gutenberg.org/cache/epub/{numero}/pg{numero}.txt"
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        destino.write_text(r.content.decode("utf-8-sig"), encoding="utf-8")
    return destino.read_text(encoding="utf-8")

crudos = {}
for nombre, (numero, _, _) in LIBROS.items():
    try:
        crudos[nombre] = descargar(numero)
        print(f"{nombre}: {len(crudos[nombre]):,} caracteres")
    except Exception as e:
        print(f"{nombre}: no se pudo descargar ({e}). Se usará la copia limpia del repositorio.")

### ¿Qué hay en un archivo de Gutenberg?

Veamos el principio de *La Navidad en las montañas*:

In [ ]:
print(crudos["altamirano_navidad_1871"][:3000])

Antes de la novela hay mucho que **no** queremos en el corpus:

- El **encabezado de Gutenberg**, en inglés, con la licencia. Al final del archivo hay otro aún más largo.
- Esta edición es un **libro escolar estadounidense de 1904**. Trae prefacio, introducción, notas al pie y un vocabulario, todo en inglés. *Doña Perfecta* tiene el mismo problema.
- **Marcas de formato**: `_cursiva_`, `#negrita#`, `=versalitas=`.
- **Saltos de línea** a mitad de oración, porque el texto se cortó a 70 caracteres.

Busquemos, por ejemplo, las notas al pie:

In [ ]:
notas = re.findall(r"\[Footnote[^\]]*\]", crudos["altamirano_navidad_1871"])
print(len(notas), "notas al pie")
print(notas[0])

Si no las quitamos, *the* sería una de las palabras más frecuentes de Altamirano.

### Limpiar

La función `limpiar()` hace todo esto paso a paso. Cada línea de código resuelve uno de los problemas de la lista anterior.

In [ ]:
def limpiar(texto, inicio, fin):
    texto = texto.replace("\r\n", "\n")

    # 1. Quitar encabezado y licencia de Gutenberg
    texto = texto.split("*** START OF")[1].split("***", 1)[1]
    texto = texto.split("*** END OF")[0]

    # 2. Quedarnos sólo con la novela (entre las líneas de inicio y fin)
    lineas = texto.split("\n")
    i = next(n for n, l in enumerate(lineas) if re.fullmatch(inicio, l.strip()))
    f = len(lineas)
    if fin:
        f = next(n for n, l in enumerate(lineas[i:], i) if re.fullmatch(fin, l.strip()))
    texto = "\n".join(lineas[i:f])

    # 3. Quitar notas al pie y llamadas a notas como [20]
    texto = re.sub(r"\[Footnote[^\]]*\]", "", texto)
    texto = re.sub(r"\[\d+\]", "", texto)

    # 4. Quitar marcas de formato
    texto = re.sub(r"[_#=]", "", texto)

    # 5. Unir las líneas cortadas: todo el espacio en blanco se vuelve un solo espacio
    return re.sub(r"\s+", " ", texto).strip()

El paso 5 elimina también la separación entre párrafos. Lo hacemos a propósito: spaCy convierte cada salto de línea en un *token* de espacio, y esos *tokens* vacíos estorbarían en las concordancias.

Aplicamos la función a las cinco novelas y guardamos los resultados en `datos/novelas_es/`. Si una descarga falló, usamos la copia del repositorio.

In [ ]:
for nombre, (numero, inicio, fin) in LIBROS.items():
    archivo = DATOS / f"{nombre}.txt"
    if nombre in crudos:
        archivo.write_text(limpiar(crudos[nombre], inicio, fin), encoding="utf-8")
    elif not archivo.exists():
        r = requests.get(RESPALDO + archivo.name, timeout=60)
        r.raise_for_status()
        archivo.write_text(r.text, encoding="utf-8")
    texto = archivo.read_text(encoding="utf-8")
    n_the = len(re.findall(r"\bthe\b", texto))
    print(f"{nombre:25} {len(texto.split()):>8,} palabras   'the': {n_the}")

Ya no queda ningún *the*. Siempre conviene comprobar la limpieza con una búsqueda así. Si el corpus fuera de otra lengua, buscaríamos otras palabras que no deberían estar.

---
## 3. Un modelo preentrenado: spaCy

CLiC sólo nos daba las palabras. Para estudiar los ojos en español queremos más información:

- el **lema**, para encontrar *ojo* y *ojos* con una sola búsqueda;
- la **categoría gramatical** (*part of speech*, POS), para distinguir un artículo de un pronombre;
- las **relaciones sintácticas**, para saber de quién son los ojos.

Anotar todo esto a mano es imposible en un corpus de 400 000 palabras. Usamos un **modelo preentrenado**: un programa que ya aprendió a anotar a partir de un corpus anotado por personas.

`es_core_news_sm` es el modelo pequeño de spaCy para el español. Veamos con qué datos se entrenó:

In [ ]:
nlp = spacy.load("es_core_news_sm")
[fuente["name"] for fuente in nlp.meta["sources"]]

**UD Spanish AnCora** es un corpus de unas 500 000 palabras, sobre todo **noticias** de periódicos españoles de finales del siglo XX, anotado a mano con el esquema de **Universal Dependencies** (UD). WikiNER sirve para reconocer nombres propios.

Retengamos esto: el modelo aprendió con prensa española moderna, y lo vamos a usar con novelas del siglo XIX, dos de ellas mexicanas.

La función `analizar()` muestra en una tabla lo que spaCy dice de cada palabra:

In [ ]:
def analizar(oracion):
    doc = nlp(oracion)
    return pd.DataFrame(
        [(t.text, t.lemma_, t.pos_, t.dep_, t.head.text, str(t.morph)) for t in doc],
        columns=["palabra", "lema", "POS", "relación", "núcleo", "rasgos"])

analizar("Se le llenaron los ojos de lágrimas.")

Las columnas:

- **POS** usa las categorías de UD: `DET` (determinante), `NOUN` (sustantivo), `PRON` (pronombre), `VERB`, `ADP` (preposición)…
- **relación** y **núcleo** forman el análisis de dependencias: *ojos* es el sujeto (`nsubj`) de *llenaron*, y *le* es un objeto (`obj`) en caso **dativo** (`Case=Dat` en los rasgos). El poseedor de los ojos está en ese clítico.

### ✅ Ejercicio 1 (resuelto)

Analiza *Cerró los ojos y miró hacia la ventana.* ¿Qué relación le asigna spaCy a *ojos*? ¿Es correcta?

In [ ]:
analizar("Cerró los ojos y miró hacia la ventana.")

spaCy dice que *los ojos* es el **sujeto** (`nsubj`) de *cerró*. Es un error: es el objeto directo. Quien cierra los ojos es un sujeto tácito.

El error tiene una explicación probable. El español permite sujetos pospuestos (*llegaron los invitados*), y en AnCora no abundan oraciones narrativas con sujeto tácito y partes del cuerpo. Un modelo preentrenado no "sabe" gramática: reproduce los patrones de los datos con que se entrenó.

---
## 4. Anotar el corpus

FlexiConc incluye `TextImport`, que hace tres cosas a la vez:

1. Lee los archivos de texto.
2. Los anota con spaCy (`use_spacy=True`). Pedimos lemas (`lemma=True`) y categorías (`pos=True`).
3. Guarda todo en una pequeña base de datos SQLite, donde luego se pueden hacer búsquedas.

Para ir más rápido, `TextImport` no usa el análisis de dependencias. Procesar las cinco novelas tarda alrededor de un minuto.

In [ ]:
archivos = sorted(DATOS.glob("*.txt"))

corpus = TextImport(DATOS / "corpus.sqlite")
corpus.load_files(archivos, shorten_paths=True, use_spacy=True,
                  spacy_model="es_core_news_sm", lemma=True, pos=True)
corpus

`corpus.tokens()` devuelve todas las palabras con sus anotaciones como un DataFrame. `file_id` dice de qué novela viene cada palabra y `s_id`, de qué oración.

In [ ]:
tokens = corpus.tokens()
tokens.iloc[1000:1015]

---
## 5. La concordancia de *ojo*

Las búsquedas usan una notación parecida a la de **CQP**, el lenguaje de consulta de CQPweb y Sketch Engine. Cada palabra de la búsqueda va entre corchetes con una condición:

| Búsqueda | Encuentra |
|---|---|
| `[word="ojos"]` | la forma exacta *ojos* |
| `[word="ojos" %c]` | *ojos*, *Ojos*, *OJOS* (`%c` ignora mayúsculas) |
| `[lemma="ojo"]` | todas las formas del lema: *ojo*, *ojos* |
| `[lemma="cerrar"] [word="los"] [lemma="ojo"]` | una secuencia de tres palabras |

Comparemos la búsqueda por forma con la búsqueda por lema:

In [ ]:
por_forma = corpus.concordance_from_query('[word="ojos" %c]', context_size=(20, 20))
C = corpus.concordance_from_query('[lemma="ojo"]', context_size=(20, 20))
print("forma 'ojos':", por_forma.root.line_count)
print("lema 'ojo':  ", C.root.line_count)

Desde aquí todo funciona como en el cuaderno 07. `C` es un objeto `Concordance` con su tabla de metadatos, su tabla de *tokens* y su árbol de análisis. La diferencia es que ahora los *tokens* también tienen `lemma` y `pos`.

In [ ]:
C.tokens.head()

In [ ]:
muestra = C.root.add_subset_node(("Random Sample", {"sample_size": 15, "seed": 42}))
show_kwic(muestra, metadata_columns=["file.path"])

---
## 6. ¿*His eyes* o *los ojos*?

Agrupamos por la palabra en L1, igual que con Dickens. Ahora usamos `locale_str="es"` para que el orden alfabético siga las reglas del español (la *ñ* después de la *n*, por ejemplo).

In [ ]:
orden_izq = C.root.add_arrangement_node(
    grouping=None,
    ordering=[
        ("Sort by Token-Level Attribute",
         {"tokens_attribute": "word", "sorting_scope": "left", "offset": 0,
          "case_sensitive": False, "reverse": False, "backwards": False, "locale_str": "es"})
    ])

por_L1 = orden_izq.add_arrangement_node(
    grouping=("Partition by Ngrams", {"positions": [-1], "tokens_attribute": "word"}),
    ordering=[])
show_kwic(por_L1, n=5, n_groups=10)

Recuerda que cada grupo aparece plegado: haz clic en **▶** para ver sus líneas. Armamos la tabla de frecuencias como en el cuaderno 07:

In [ ]:
def tabla_de_grupos(nodo, total):
    filas = [(g["label"].strip("(),'"), len(g["line_ids"])) for g in nodo.grouping_result["partitions"]]
    df = pd.DataFrame(filas, columns=["grupo", "lineas"])
    df["porcentaje"] = (100 * df["lineas"] / total).round(1)
    return df

frec_L1 = tabla_de_grupos(por_L1, C.root.line_count)
frec_L1.head(10)

### ✅ Ejercicio 2 (resuelto)

1. ¿Qué porcentaje de líneas tiene un **posesivo** en L1 (*sus, su, mis, mi, tus, tu, nuestros, vuestros*)? Compáralo con el 60 % de Dickens.
2. ¿Y con artículo definido (*los, el*)?

In [ ]:
posesivos = ["sus", "su", "mis", "mi", "tus", "tu", "nuestros", "vuestros"]
articulos = ["los", "el"]
print("posesivo:", round(frec_L1[frec_L1["grupo"].isin(posesivos)]["porcentaje"].sum(), 1))
print("artículo:", round(frec_L1[frec_L1["grupo"].isin(articulos)]["porcentaje"].sum(), 1))

Alrededor del 30 % lleva posesivo y cerca de la mitad, artículo definido. En Dickens la relación era la contraria. Ésta es la diferencia tipológica que esperábamos: en español, las partes del cuerpo suelen llevar **artículo**, y el poseedor se recupera por otra vía.

Aun así, un 30 % de posesivos no es poco. Al revisar esas líneas vas a ver muchos casos con adjetivo antepuesto (*sus hermosos ojos*, *vuestros ojos*), un recurso más literario que coloquial.

---
## 7. ¿De quién son los ojos?

Si no hay posesivo, ¿cómo sabemos de quién son los ojos? Hay dos vías principales:

- **El sujeto**: en *cerró los ojos*, son los ojos de quien los cierra.
- **Un clítico dativo**: en *se **le** llenaron los ojos*, son los ojos de la persona a la que remite *le*.

### Los verbos

Nos quedamos con las líneas que tienen *los* en L1 y las agrupamos por el **lema** en L2. Así *abrió*, *abría* y *abriendo* caen en el mismo grupo.

In [ ]:
con_los = por_L1.add_subset_node(
    ("Select by Token-Level String Attribute",
     {"search_terms": ["los"], "tokens_attribute": "word", "offset": -1}))

L2_lema = con_los.add_arrangement_node(
    grouping=("Partition by Ngrams", {"positions": [-2], "tokens_attribute": "lemma"}),
    ordering=[])
show_kwic(L2_lema, n=4, n_groups=12)

Encontramos las mismas acciones que en Dickens: **abrir**, **cerrar**, **alzar**, **fijar**, **bajar**, **apartar**. También aparece *con los ojos*, el equivalente de *with his eyes*.

### Los clíticos

Ahora buscamos clíticos en las seis palabras anteriores a *ojos*. Usamos el *KWIC Grouper Ranker* del cuaderno 07: pone primero las líneas con más clíticos en esa ventana.

In [ ]:
cliticos = ["me", "te", "le", "nos", "os", "les", "se"]

por_clitico = con_los.add_arrangement_node(
    grouping=None,
    ordering=[
        ("KWIC Grouper Ranker",
         {"search_terms": cliticos, "tokens_attribute": "word", "mode": "literal",
          "case_sensitive": False, "window_start": -6, "window_end": -1,
          "include_node": False, "count_types": False})
    ])
show_kwic(por_clitico, n=25)

### ✅ Ejercicio 3 (resuelto)

Cuenta cuántas líneas con *los ojos* tienen un clítico en esa ventana. ¿Qué vía es más común para identificar al poseedor?

In [ ]:
T = C.tokens
lineas_los = set(T[(T["offset"] == -1) & (T["word"].str.lower() == "los")]["line_id"])
ventana = T[T["offset"].between(-6, -1) & T["line_id"].isin(lineas_los)]
con_clitico = set(ventana[ventana["word"].str.lower().isin(cliticos)]["line_id"])
print(len(con_clitico), "de", len(lineas_los))

Sólo una de cada nueve, más o menos. En estas novelas el poseedor casi siempre se recupera del **sujeto** (*abrió los ojos*, *alzó los ojos*). Además, no todos los clíticos encontrados son dativos de posesión: algunos son reflexivos o pertenecen a otro verbo de la oración. Para separarlos haría falta el análisis de dependencias, y eso nos lleva a la sección 9.

---
## 8. Comparar novelas

Los metadatos de cada línea incluyen `file.path`, el archivo del que viene. *Partition by Metadata Attribute* agrupa por esa columna.

In [ ]:
por_novela = C.root.add_arrangement_node(
    grouping=("Partition by Metadata Attribute", {"metadata_attribute": "file.path"}),
    ordering=[])
show_kwic(por_novela, n=3)

Como vimos en el cuaderno 07, las frecuencias absolutas engañan si las novelas tienen tamaños distintos. Normalizamos: apariciones por cada 10 000 palabras.

In [ ]:
nombres = {i: archivo.stem for i, archivo in enumerate(archivos)}
tamanos = tokens[tokens["pos"] != "PUNCT"].groupby("file_id").size().rename(index=nombres)
ojos = C.metadata["file.path"].str.removesuffix(".txt").value_counts()

comparacion = pd.DataFrame({"palabras": tamanos, "ojo": ojos})
comparacion["por_10mil"] = (10_000 * comparacion["ojo"] / comparacion["palabras"]).round(1)
comparacion.sort_values("por_10mil", ascending=False)

### Para discutir

- *Marianela* tiene con mucho la frecuencia más alta. La novela trata de Pablo, un joven ciego que recupera la vista. Aquí la frecuencia de una palabra refleja **el tema** de la obra, no el estilo del autor.
- Las dos novelas mexicanas tienen las frecuencias más bajas. ¿Podemos concluir algo sobre "la novela mexicana"? Con dos obras, no: necesitaríamos muchas más para separar el efecto del país del efecto de cada autor y cada tema.

---
## 9. ¿Qué tan bien anotó spaCy?

Un modelo preentrenado ahorra muchísimo trabajo, pero sus errores pasan a nuestros resultados. Siempre hay que revisar una muestra.

### Errores en *ojo*

Busquemos las apariciones de *ojo* que spaCy **no** etiquetó como sustantivo:

In [ ]:
errores = tokens[(tokens["lemma"] == "ojo") & (tokens["pos"] != "NOUN")]
print(len(errores), "de", (tokens["lemma"] == "ojo").sum())
for cpos in errores["cpos"]:
    contexto = " ".join(tokens["word"].iloc[cpos - 5 : cpos + 3])
    print(f"{tokens['pos'].iloc[cpos]:5} {contexto}")

Casi todos tienen un **adjetivo antepuesto**: *despavoridos ojos*, *sagaces ojos*. En la prensa, el adjetivo calificativo casi siempre va después del sustantivo. El modelo vio pocos ejemplos de este orden, que es típico de la prosa literaria, y se confunde. Los demás casos llevan *vuestros*, una forma que casi no aparece en la prensa.

### La ortografía del siglo XIX

*Monja y casada* se publicó en 1868, con la ortografía de la época: *á* como preposición, *fué*, *dió*, *habia* sin tilde. Veamos qué hace spaCy con eso:

In [ ]:
for palabra in ["á", "habia", "dió", "fué"]:
    x = tokens[tokens["word"] == palabra]
    print(f"{palabra:6} {len(x):5} apariciones | POS: {x['pos'].value_counts().head(3).to_dict()} | lema: {x['lemma'].value_counts().head(2).to_dict()}")

Los resultados son malos. La preposición *á* sólo se reconoce como preposición en poco más de un tercio de los casos; a veces se etiqueta como verbo con el lema inventado *ar*. *Dió* recibe el lema *deír*, y *habia*, el lema *habiar*.

Esto importa para la concordancia: si agrupamos por lema en L2, *dió los ojos* no caería con *dar*.

### Adaptar el texto al modelo

No podemos reentrenar el modelo en clase, pero sí podemos **acercar el texto a lo que el modelo conoce**. Normalizamos las formas antiguas más frecuentes antes de anotar:

In [ ]:
MODERNIZAR = {
    r"\bá\b": "a", r"\bÁ\b": "A",
    r"\bfué\b": "fue", r"\bFué\b": "Fue",
    r"\bdió\b": "dio", r"\bvió\b": "vio",
    r"\bhabia\b": "había", r"\bhabian\b": "habían",
}

def modernizar(texto):
    for antigua, moderna in MODERNIZAR.items():
        texto = re.sub(antigua, moderna, texto)
    return texto

ejemplo = "No habia luz, y él dió un paso y fué á buscarla."
pd.concat([analizar(ejemplo)[["palabra", "lema", "POS"]],
           analizar(modernizar(ejemplo))[["palabra", "lema", "POS"]]],
          axis=1, keys=["original", "modernizado"])

Con el texto modernizado, *había*, *dio*, *fue* y *a* reciben el lema y la categoría correctos.

### ✅ Ejercicio 4 (resuelto)

Moderniza *Monja y casada*, vuelve a anotar el corpus y comprueba cuántas *a* se reconocen ahora como preposición.

In [ ]:
MODERNOS = DATOS / "modernizados"
MODERNOS.mkdir(exist_ok=True)
for archivo in archivos:
    (MODERNOS / archivo.name).write_text(modernizar(archivo.read_text(encoding="utf-8")), encoding="utf-8")

corpus_mod = TextImport(DATOS / "corpus_modernizado.sqlite")
corpus_mod.load_files(sorted(MODERNOS.glob("*.txt")), shorten_paths=True, use_spacy=True,
                      spacy_model="es_core_news_sm", lemma=True, pos=True)
tokens_mod = corpus_mod.tokens()

monja = list(nombres.values()).index("rivapalacio_monja_1868")
for nombre, tk in [("original", tokens), ("modernizado", tokens_mod)]:
    a = tk[(tk["file_id"] == monja) & (tk["word"].str.lower().isin(["a", "á"]))]
    print(f"{nombre:12} 'a' como preposición: {(a['pos'] == 'ADP').mean():.0%}")

La proporción sube muchísimo. Con unas cuantas reglas corregimos miles de etiquetas.

Pero fíjate en lo que tuvimos que hacer: **saber** qué formas antiguas había, escribir las reglas y revisar que no rompieran nada. Ese conocimiento viene de la lingüística, no del modelo.

---
## 10. Lo que aprendimos y lo que sigue

- Un corpus "disponible" casi nunca está listo: hay que **limpiarlo** y **comprobar** la limpieza.
- Un modelo preentrenado nos da lemas y categorías gratis, pero **hereda los sesgos de sus datos de entrenamiento**: prensa moderna de España. Falla con el orden literario del adjetivo, con la ortografía del XIX y con algunas construcciones.
- Con unas reglas de normalización podemos **acercar** nuestros textos al modelo.

El español es una de las lenguas con más recursos del mundo: hay corpus anotados, modelos de varios tamaños y décadas de trabajo. En el cuaderno 09 veremos qué pasa con una lengua que **no** tiene modelo de spaCy: el náhuatl.

---
## Referencias

- de Marneffe, M.-C., Manning, C. D., Nivre, J. y Zeman, D. (2021). Universal Dependencies. *Computational Linguistics*, 47(2), 255–308.
- Evert, S. et al. (2025). *Reading Concordances*. Curso en ESSLLI 2025, Bochum. Materiales: [github.com/reading-concordances/teaching](https://github.com/reading-concordances/teaching).
- Honnibal, M., Montani, I., Van Landeghem, S. y Boyd, A. (2020). *spaCy: Industrial-strength Natural Language Processing in Python*. [spacy.io](https://spacy.io).
- Taulé, M., Martí, M. A. y Recasens, M. (2008). AnCora: Multilevel Annotated Corpora for Catalan and Spanish. *Proceedings of LREC 2008*.
- Textos: Project Gutenberg, números 10825, 15725, 17340, 18005 y 49560.